In [14]:
import pandas as pd
import PyPDF2
import pdfplumber
import re
import os

In [16]:
#os.getcwd()
os.chdir("C:\\Users\\Administrator\\Desktop\\analysis\\Test folder 2")
os.getcwd()

'C:\\Users\\Administrator\\Desktop\\analysis\\Test folder 2'

In [17]:
import pandas as pd
import PyPDF2
import pdfplumber
import re
import os


folder_path = "C:\\Users\\Administrator\\Desktop\\analysis\\Test folder 2"

pdf_files = [
    f for f in os.listdir(folder_path)
    if f.lower().endswith(".pdf")
]

pdf_combined = pd.DataFrame()

# Store all dataframes for each PDF to create separate sheets
all_metadata = []
all_summary = []
all_executive = []

for pdf_file in pdf_files:
    try:
        print(f"\n{'='*100}")
        print(f"Processing: {pdf_file}")
        print(f"{'='*100}")
        
        # Get filename for source tracking
        pdf_filename = os.path.basename(pdf_file)
        
        # Create a PDF reader object
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        # Get the first page (page 0)
        first_page = pdf_reader.pages[0]

        # Extract text from the first page
        first_page_text = first_page.extract_text()

        # Defining regex patterns for each field
        regex_patterns = {
            'Site': r"SITE\s*(.*?)\s*CLIENT",
            'Our reference': r"OUR REFERENCE\s*([^\n]+)",
            'Your reference': r"YOUR REFERENCE\s*([^\n]+)",
            'Visit Date': r"VISIT DATE\s*([^\n]+)",
            'Scope of Work': r"SCOPE OF WORK\s*(.*?)\s*(?=-)",
            'Notes': r"SCOPE OF WORK\s*.*?-\s*(.*?)\."
        }

        # Initialize an empty dictionary to store extracted data
        extracted_data = {}

        # Loop through each field and pattern to extract information
        for field, pattern in regex_patterns.items():
            match = re.search(pattern, first_page_text, re.DOTALL)
            if match:
                if field == 'Site':
                    site_block = match.group(1).strip()
                    site_lines = site_block.split('\n')
                    cleaned_site = site_lines[0].strip()
                    if len(site_lines) > 1 and site_lines[1].strip():
                        cleaned_site += ' ' + site_lines[1].strip()
                    extracted_data[field] = cleaned_site.replace('\n', ' ')
                elif field == 'Scope of Work':
                    extracted_data[field] = match.group(1).strip().replace('\n', ' ')
                else:
                    extracted_data[field] = match.group(1).strip()
            else:
                extracted_data[field] = 'N/A'

        # Convert the dictionary to a DataFrame
        df_page1 = pd.DataFrame([extracted_data])

        df_page1.rename(columns={
            "Site": "Site_Name",
            "Our reference": "Report_Ref",
            "Your reference": "Client_Ref",
            "Scope of Work": "Scope_of_Work",
            "Visit Date": "Visit_Date"    
        }, inplace=True)
        
        # Add source to metadata
        df_page1["pdf_source"] = pdf_filename

        # SECTION 1: Extract Summary Table
        with pdfplumber.open(pdf_file) as pdf:
            summary_table_found = False
            all_table_data = []
            headers = None
            start_page = None
            
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                
                if "SUMMARY TABLE" in text and not summary_table_found:
                    print(f"Found SUMMARY TABLE starting on page {page_num}")
                    summary_table_found = True
                    start_page = page_num
                
                if summary_table_found:
                    if "LOGBOOK" in text:
                        print(f"Found LOGBOOK on page {page_num} - stopping table extraction")
                        break
                    
                    tables = page.extract_tables()
                    
                    if tables:
                        for table in tables:
                            if table:
                                if headers is None:
                                    headers = table[0]
                                    all_table_data.extend(table[1:])
                                else:
                                    if table[0] == headers:
                                        all_table_data.extend(table[1:])
                                    else:
                                        all_table_data.extend(table)
                    
                    print(f"  Processed page {page_num} - Total rows collected: {len(all_table_data)}")
            
            if summary_table_found and all_table_data:
                df = pd.DataFrame(all_table_data, columns=headers)
                df = df.dropna(how='all')
                
                df.columns = df.iloc[0]
                df = df[1:]
                df = df.reset_index(drop=True)
                
                df.columns.values[0] = 'Location'
                df.columns.values[1] = 'Asset_Name'
                df.columns = df.columns.str.replace(r'[\n\t\r]+', ' ', regex=True)
                df.columns = df.columns.str.strip()
                
                df.rename(columns={
                    "OCWS Supply Size": "CWS_Supply_C",
                    "HWS Return Size": "HWS_Return_C",
                    "Strainers Cleaned?": "Strainers_Cleaned",
                    "Before Clean": "Before_Clean",
                    "After Clean": "After_Clean",
                    "Record pipework insulation": "Record_pipework_insulation",
                    "Return Temp": "Return_Temp",
                    "Isolation Fitted?": "Isolation_Fitted",
                    "Isolation Working?": "Isolation_Working",
                    "Cold Temp to TMV": "Cold_Temp_to_TMV",
                    "Hot Temp to TMV": "Hot_Temp_to_TMV",
                    "Fail Safe Test Result": "Fail_Safe_Test_Result",
                    "Volume of discharged water.": "Volume_of_discharged_water",
                    "Blended Temp (Basin /Shower)": "Blended_Temp_(Basin/Shower)",
                    "Blended Temp (Basin/Shower)": "Blended_Temp_(Basin/Shower)",
                    "Blended Temp (Unassisted Bath)": "Blended_Temp_(Unassisted_Bath)",
                    "Blended Temp (Assisted Bath)": "Blended_Temp_(Assisted_Bath)",
                    "Blended Temp (Bidet)": "Blended_Temp_(Bidet)",
                    "CWS Supply Size": "CWS_Supply_Size",
                    "HWS Supply Size": "HWS_Supply_Size",
                    "After Photo": "After_Photo"
                }, inplace=True)
                
                df['Location'] = df['Location'].str.replace(r'[\n\t\r]+', ' ', regex=True)
                df['Location'] = df['Location'].str.replace(r'\s+', ' ', regex=True)
                df['Location'] = df['Location'].str.strip()
                
                df = df.join(df['Location'].str.split('/', expand=True).rename(columns={
                    0: "Floor",
                    1: "Ward/Department",
                    2: "Room_ID",
                    3: "Fac",
                    4: "other"
                }))

                df['Fixture'] = df['Floor'].astype(str).str.split().str[:-2].str.join(' ')
                df['Floor'] = df['Floor'].astype(str).str.split().str[-2:].str.join(' ')
                
                df = df.drop(columns=["Location", "other", "Fac"], errors='ignore')
                
                df['Room_Name'] = df['Room_ID'].str.split().str[1:].str.join(' ')
                df['Room_ID'] = df['Room_ID'].astype(str).str.split().str[0]

                # Clean temperature columns
                df['Return_Temp'] = df['Return_Temp'].str.replace("°C", " ")
                df['Cold_Temp_to_TMV'] = df['Cold_Temp_to_TMV'].str.replace("°C", " ")
                df['Hot_Temp_to_TMV'] = df['Hot_Temp_to_TMV'].str.replace("°C", " ")
                df['HWS_Supply_Size'] = df['HWS_Supply_Size'].str.replace("mm", " ")
                df['CWS_Supply_Size'] = df['CWS_Supply_Size'].str.replace("mm", " ")
                df['Blended_Temp_(Basin/Shower)'] = df['Blended_Temp_(Basin/Shower)'].str.replace("°C", " ")
                df['Blended_Temp_(Unassisted_Bath)'] = df['Blended_Temp_(Unassisted_Bath)'].str.replace("°C", " ")
                df['Blended_Temp_(Assisted_Bath)'] = df['Blended_Temp_(Assisted_Bath)'].str.replace("°C", " ")
                df['Blended_Temp_(Bidet)'] = df['Blended_Temp_(Bidet)'].str.replace("°C", " ")
                
                df['Site_Name'] = df_page1["Site_Name"].iloc[0]
                df['Visit_Dates'] = df_page1["Visit_Date"].iloc[0]
                df['Report_Ref'] = df_page1["Report_Ref"].iloc[0]
                df['Client_Ref'] = df_page1["Client_Ref"].iloc[0]
                df['Scope_of_Work'] = df_page1["Scope_of_Work"].iloc[0]

                column_order = ['Site_Name', 'Visit_Dates', 'Report_Ref', 'Client_Ref', 'Scope_of_Work',
                    'Asset_Name', 'Floor', 'Ward/Department', 'Room_ID', 'Room_Name', 'Fixture', 'HWS_Supply_Size', 'CWS_Supply_Size',
                    'Strainers_Cleaned', 'Before_Clean', 'After_Clean', 'Record_pipework_insulation', 'Return_Temp', 'Isolation_Working',
                    'Isolation_Fitted', 'Cold_Temp_to_TMV', 'Hot_Temp_to_TMV', 'Fail_Safe_Test_Result',
                    'Volume_of_discharged_water', 'Blended_Temp_(Basin/Shower)', 'Blended_Temp_(Unassisted_Bath)',
                    'Blended_Temp_(Assisted_Bath)', 'Blended_Temp_(Bidet)', 'After_Photo']
                df = df[column_order]
                
                # Add source
                df["pdf_source"] = pdf_filename
            else:
                print("No summary table found, creating empty dataframe")
                df = pd.DataFrame()

        # SECTION 2: Extract Executive Summary (3-column tables)
        with pdfplumber.open(pdf_file) as pdf:
            exec_summary = False
            works_not_completable = False
            data = []
            first_table_skipped = False
            
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                
                if "EXECUTIVE SUMMARY" in text:
                    exec_summary = True
                    print(f"EXECUTIVE SUMMARY found on page {page_num}")
                
                if "Works not completable" in text:
                    works_not_completable = True
                
                if exec_summary:
                    if "SUMMARY TABLE" in text:
                        print(f"SUMMARY TABLE found on page {page_num} - stopping")
                        break
                    
                    tables = page.extract_tables()
                    
                    for table in tables:
                        if table and len(table) > 0:
                            if not first_table_skipped:
                                print(f"  Skipping first table on page {page_num}")
                                first_table_skipped = True
                                continue
                            
                            if works_not_completable:
                                num_cols = len(table[0]) if table[0] else 0
                                if num_cols == 3:
                                    data.extend(table)
                                else:
                                    print(f"  Skipping {num_cols}-column table in Works not completable on page {page_num}")
                            else:
                                print(f"  Extracting table on page {page_num} ({len(table[0])} columns)")
                                data.extend(table)
            
            if data:
                df_exe_1 = pd.DataFrame(data)
                
                if df_exe_1.shape[1] >= 3:
                    df_exe_1.rename(columns={
                        0: "location",
                        1: "Issue_Summary",
                        2: "Severity"
                    }, inplace=True)
                
                df_exe_1 = df_exe_1[df_exe_1['Issue_Summary'].astype(bool)]
                df_exe_1['location'] = df_exe_1['location'].ffill()
            else:
                print("No executive summary data found")
                df_exe_1 = pd.DataFrame(columns=["location", "Issue_Summary", "Severity"])

        # SECTION 3: Extract Works not completable (2-column tables)
        with pdfplumber.open(pdf_file) as pdf:
            wrk_not_completable = False
            data = []
            
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                
                if "Works not completable" in text:
                    wrk_not_completable = True
                    # print(f"Works not completable found on page {page_num}")
                
                if wrk_not_completable:
                    if "SUMMARY TABLE" in text:
                        # print(f"summary table {page_num} - stopping")
                        break
                    
                    tables = page.extract_tables()
                    
                    for table in tables:
                        if wrk_not_completable:
                            num_cols = len(table[0]) if table[0] else 0
                            if num_cols == 2:
                                data.extend(table)
            
            if data:
                df_exe_2 = pd.DataFrame(data)
                df_exe_2.rename(columns={
                    0: "location",
                    1: "Issue_Summary",
                }, inplace=True)
            else:
                print("No works not completable data found")
                df_exe_2 = pd.DataFrame(columns=["location", "Issue_Summary"])

        # SECTION 4: Combine and merge
        appended_data = pd.concat([df_exe_1, df_exe_2], ignore_index=True)
        
        if not appended_data.empty:
            appended_data["Asset_Name"] = appended_data['location'].str.extract(r'(TMV_\S*)', expand=False)
            appended_data = appended_data.drop(columns=["location"], errors='ignore')
            
            if not df.empty:
                merged_left = pd.merge(appended_data, df, on=['Asset_Name'], how='left')
                
                # Define column order (removed duplicate Asset_Name)
                column_order = ["Asset_Name", "Visit_Dates", "Report_Ref", "Client_Ref", "Scope_of_Work",
                    "Floor", "Ward/Department", "Room_ID", "Room_Name", "Fixture", "Severity", "Issue_Summary"]
                
                # Only select columns that exist
                available_columns = [col for col in column_order if col in merged_left.columns]
                merged_left = merged_left[available_columns]
                
                merged_left["pdf_source"] = pdf_filename
                
                # Append to combined dataset
                pdf_combined = pd.concat([pdf_combined, merged_left], ignore_index=True, sort=False)
                
                # Store individual dataframes for Excel output
                all_metadata.append(df_page1)
                all_summary.append(df)
                all_executive.append(merged_left)
                
                # Save individual PDF output to Excel with 3 sheets
                # output_filename = f'Final_output_{pdf_filename.replace(".pdf", "")}.xlsx'
                # with pd.ExcelWriter(output_filename) as writer:
                #     df_page1.to_excel(writer, sheet_name='Metadata', index=False)
                #     df.to_excel(writer, sheet_name='Summary_data', index=False)
                #     merged_left.to_excel(writer, sheet_name='Executive_data', index=False)
                
                # print(f"Individual file saved: {output_filename}")
            else:
                print("Skipping merge - summary table is empty")
        else:
            print("Skipping - no executive summary data found")
    
    except Exception as e:
        print(f"ERROR processing {pdf_file}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue


# Save combined CSV and Excel
# pdf_combined.to_csv('combined_pdfs_output.csv', index=False)
# pdf_combined.to_excel('combined_pdfs_output.xlsx', index=False)


# Create one master Excel file with all PDFs combined
if all_metadata and all_summary and all_executive:
    combined_metadata = pd.concat(all_metadata, ignore_index=True)
    combined_summary = pd.concat(all_summary, ignore_index=True)
    combined_executive = pd.concat(all_executive, ignore_index=True)
    
    with pd.ExcelWriter('Final_output_ALL_PDFS.xlsx') as writer:
        combined_metadata.to_excel(writer, sheet_name='Metadata', index=False)
        combined_summary.to_excel(writer, sheet_name='Summary_data', index=False)
        combined_executive.to_excel(writer, sheet_name='Executive_data', index=False)
    


print("\n" + "="*100)
print("PROCESSING COMPLETE!")
print("="*100)


Processing: Guy_s_Hospital_Borough_Wing_442421_1.pdf
Found SUMMARY TABLE starting on page 9
  Processed page 9 - Total rows collected: 15
  Processed page 10 - Total rows collected: 36
  Processed page 11 - Total rows collected: 45
Found LOGBOOK on page 12 - stopping table extraction
EXECUTIVE SUMMARY found on page 3
  Skipping first table on page 3
  Extracting table on page 3 (3 columns)
  Extracting table on page 3 (3 columns)
  Extracting table on page 3 (3 columns)
  Extracting table on page 4 (3 columns)
  Extracting table on page 4 (3 columns)
  Extracting table on page 4 (3 columns)
  Extracting table on page 4 (3 columns)
  Extracting table on page 4 (3 columns)
  Extracting table on page 5 (3 columns)
  Extracting table on page 5 (3 columns)
  Extracting table on page 5 (3 columns)
  Extracting table on page 5 (3 columns)
  Extracting table on page 5 (3 columns)
  Extracting table on page 6 (3 columns)
  Extracting table on page 6 (3 columns)
  Extracting table on page 6 (3 